# 项目图片生成示例

这个 notebook 用来演示项目里推荐的文生图调用方式。

推荐的几种路径：
- Python 内部直接调用：`generate_image(...)`
- Python 内部轻量模型调用：`generate_mini_image(...)`
- 前端/联调调用：`POST http://127.0.0.1:8787/api/t2i`

以后如果你在项目的 Python 代码里生成图片，优先使用前两种方式。


In [1]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    # 从当前目录开始逐级向上查找，直到找到包含 Code 目录的项目根目录。
    for candidate in [start, *start.parents]:
        if (candidate / 'Code').exists():
            return candidate
    raise RuntimeError(f'无法从当前路径定位仓库根目录：{start}')


ROOT = find_repo_root(Path.cwd().resolve())
CODE_DIR = (ROOT / 'Code').resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

TEST_OUTPUT_DIR = ROOT / 'outputs' / 'notebook_tests'
TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('项目根目录 =', ROOT)
print('代码目录 =', CODE_DIR)
print('notebook 输出目录 =', TEST_OUTPUT_DIR)


项目根目录 = F:\Documents\GitHub\AI_TRPG_616
代码目录 = F:\Documents\GitHub\AI_TRPG_616\Code
notebook 输出目录 = F:\Documents\GitHub\AI_TRPG_616\outputs\notebook_tests


In [2]:
from image_model.function_ImageGeneration import (
    generate_image,
    generate_mini_image,
    get_selected_image_model_code,
    get_selected_image_model_config,
)

model_code = get_selected_image_model_code()
model_config = get_selected_image_model_config()
base_url, model_name = model_config.resolve_endpoint(feature='text_to_image')
mini_base_url, mini_model_name = model_config.resolve_endpoint(feature='mini_version')

print('当前选中的图片模型 code =', model_code)
print('配置中的 code =', model_config.code)
print('普通模型名 =', model_name)
print('普通 base_url =', base_url)
print('mini 模型名 =', mini_model_name)
print('mini base_url =', mini_base_url)
print('API Key 环境变量名 =', model_config.api_key_env)
print('最大 prompt 长度 =', model_config.max_prompt_chars)


当前选中的图片模型 code = gemini
配置中的 code = gemini
普通模型名 = gemini-3-pro-image-preview
普通 base_url = https://generativelanguage.googleapis.com/v1beta
mini 模型名 = gemini-2.5-flash-image
mini base_url = https://generativelanguage.googleapis.com/v1beta
API Key 环境变量名 = GEMINI_API_KEY
最大 prompt 长度 = 4000


In [3]:
# 下面这些是项目里文生图最常用的参数。
# PROMPT: 你想生成什么画面。
# TRIGGER: 这次请求属于哪种触发类型，会影响最终 prompt 的包装方式。
# THEME: 视觉主题，会影响项目内置风格描述。
# SCENE_ID: 场景标识，用于缓存；相同场景可以复用已生成图片。
# CHARACTERS: 角色一致性信息，可传入角色名和外观描述。

PROMPT = 'TRPG desert ruin at dusk, broken stone gate, blowing sand, cinematic composition, no text'
TRIGGER = 'manual'
THEME = 'neon'
SCENE_ID = 'notebook-image-model-test'
CHARACTERS = []
SERVER_ENDPOINT = 'http://127.0.0.1:8787/api/t2i'
USE_HTTP_SERVER = False

PROMPT


'TRPG desert ruin at dusk, broken stone gate, blowing sand, cinematic composition, no text'

## 1. 普通模型调用：`generate_image(...)`

这是项目内最推荐的标准用法，会使用 `text_to_image` 对应的主图像模型。


In [4]:
# from IPython.display import Image, display

# result = generate_image(
#     prompt=PROMPT,
#     trigger=TRIGGER,
#     theme=THEME,
#     scene_id=SCENE_ID,
#     characters=CHARACTERS,
#     save_output=True,
#     use_cache=True,
# )

# print('提供方 =', result.provider)
# print('模型 code =', result.model_code)
# print('模型名 =', result.model_name)
# print('是否命中缓存 =', result.cached)
# print('总耗时（秒） =', result.elapsed_seconds)
# print('实际尝试次数 =', result.attempt_count)
# print('最终 prompt =', result.prompt)
# print('输出文件路径 =', result.output_path)
# print('图片 MIME 类型 =', result.mime_type)

# if result.output_path:
#     display(Image(filename=str(result.output_path)))


## 1.1 轻量模型调用：`generate_mini_image(...)`

这一格会使用 `data_ImageModel.yml` 里的 `mini_version` 配置，也就是轻量/更快的图像模型。


In [5]:
# mini_result = generate_mini_image(
#     prompt=PROMPT,
#     trigger=TRIGGER,
#     theme=THEME,
#     scene_id=f'{SCENE_ID}-mini',
#     characters=CHARACTERS,
#     save_output=True,
#     use_cache=True,
# )

# print('提供方 =', mini_result.provider)
# print('模型 code =', mini_result.model_code)
# print('模型名 =', mini_result.model_name)
# print('是否命中缓存 =', mini_result.cached)
# print('总耗时（秒） =', mini_result.elapsed_seconds)
# print('实际尝试次数 =', mini_result.attempt_count)
# print('最终 prompt =', mini_result.prompt)
# print('输出文件路径 =', mini_result.output_path)
# print('图片 MIME 类型 =', mini_result.mime_type)

# if mini_result.output_path:
#     display(Image(filename=str(mini_result.output_path)))


## 1.2 普通模型 vs 轻量模型测速

下面这一格会用同一 prompt 对比普通图像模型和 mini 图像模型的耗时。
为了避免缓存影响，这里强制 `use_cache=False`。


In [6]:
# import time

# def time_image_call(fn, **kwargs):
#     start = time.perf_counter()
#     result = fn(**kwargs)
#     elapsed = time.perf_counter() - start
#     return elapsed, result

# normal_elapsed, normal_bench = time_image_call(
#     generate_image,
#     prompt=PROMPT,
#     trigger=TRIGGER,
#     theme=THEME,
#     scene_id=f'{SCENE_ID}-bench-normal',
#     characters=CHARACTERS,
#     save_output=True,
#     use_cache=False,
# )

# mini_elapsed, mini_bench = time_image_call(
#     generate_mini_image,
#     prompt=PROMPT,
#     trigger=TRIGGER,
#     theme=THEME,
#     scene_id=f'{SCENE_ID}-bench-mini',
#     characters=CHARACTERS,
#     save_output=True,
#     use_cache=False,
# )

# print('Normal model:', normal_bench.model_name, f'{normal_elapsed:.2f}s')
# print('Mini model:', mini_bench.model_name, f'{mini_elapsed:.2f}s')


In [7]:
import os
from openai import OpenAI


# 请确保您已将 API Key 存储在环境变量 DOUBAO_API_KEY 中 
# 初始化Ark客户端，从环境变量中读取您的API Key 
client = OpenAI( 
    # 此为默认路径，您可根据业务所在地域进行配置 
    base_url="https://ark.cn-beijing.volces.com/api/v3", 
    # 从环境变量中获取您的 API Key。此为默认方式，您可根据需要进行修改 
    api_key=os.environ.get("DOUBAO_API_KEY"), 
) 
 
imagesResponse = client.images.generate( 
    model="doubao-seedream-5-0-260128", 
    prompt="星际穿越，黑洞，黑洞里冲出一辆快支离破碎的复古列车，抢视觉冲击力，电影大片，末日既视感，动感，对比色，oc渲染，光线追踪，动态模糊，景深，超现实主义，深蓝，画面通过细腻的丰富的色彩层次塑造主体与场景，质感真实，暗黑风背景的光影效果营造出氛围，整体兼具艺术幻想感，夸张的广角透视效果，耀光，反射，极致的光影，强引力，吞噬",
    size="2K",
    response_format="url",
    extra_body={
        "watermark": True,
    },
) 
 
print(imagesResponse.data[0].url)

https://ark-acg-cn-beijing.tos-cn-beijing.volces.com/doubao-seedream-5-0/021773646307004a8d0ee29d045ba60b66837dfbe7df0e9a07a39_0.jpeg?X-Tos-Algorithm=TOS4-HMAC-SHA256&X-Tos-Credential=AKLTYWJkZTExNjA1ZDUyNDc3YzhjNTM5OGIyNjBhNDcyOTQ%2F20260316%2Fcn-beijing%2Ftos%2Frequest&X-Tos-Date=20260316T073218Z&X-Tos-Expires=86400&X-Tos-Signature=679bc07369c9a7cd89a4276f3330c7a69e1a4098d4d85d38375e60dc5da0cba1&X-Tos-SignedHeaders=host


## 1.2.1 Doubao / Seedream 普通模型调用

这一格会走项目封装的 `generate_image(...)`，并显式覆盖模型名为 `doubao-seedream-5-0-260128`，方便直接验证仓库侧的豆包生图链路。


In [8]:
doubao_result = generate_image(
    prompt=PROMPT,
    trigger=TRIGGER,
    theme=THEME,
    scene_id=f'{SCENE_ID}-doubao',
    characters=CHARACTERS,
    model_code='doubao',
    model_name='doubao-seedream-5-0-260128',
    save_output=True,
    use_cache=False,
    debug=True,
)

print('提供方 =', doubao_result.provider)
print('模型 code =', doubao_result.model_code)
print('模型名 =', doubao_result.model_name)
print('是否命中缓存 =', doubao_result.cached)
print('总耗时（秒） =', doubao_result.elapsed_seconds)
print('实际尝试次数 =', doubao_result.attempt_count)
print('输出文件路径 =', doubao_result.output_path)

if doubao_result.output_path:
    display(Image(filename=str(doubao_result.output_path)))


[seedream-image] model=doubao-seedream-5-0-260128 prompt_chars=205 reference_count=0 retries=3 read_timeout=90s


RuntimeError: HTTP 400: The parameter `size` specified in the request is not valid: image size must be at least 3686400 pixels. Request id: 021773646339665e072dffc7e50b3adb3a3ea40a5b9ff71ae4b54

## 1.2.2 Doubao / Seedream 官方 SDK 直连调用

这一格直接使用 `openai` SDK 调豆包图片接口，和你提供的可运行示例保持一致，便于判断问题是在项目封装层还是在账号/模型侧。


In [ ]:
# import os
# from openai import OpenAI


# client = OpenAI(
#     base_url='https://ark.cn-beijing.volces.com/api/v3',
#     api_key=os.environ.get('DOUBAO_API_KEY'),
# )

# images_response = client.images.generate(
#     model='doubao-seedream-5-0-260128',
#     prompt=PROMPT,
#     size='2K',
#     response_format='url',
#     extra_body={
#         'watermark': True,
#     },
# )

# print(images_response.data[0].url)


## 1.3 参考图生成示例

如果你想让 Gemini 参考已有图片继续生成，可以传入 `reference_images`。

目前支持的参考图输入格式：
- 本地文件路径字符串
- `Path(...)`
- `{"path": "..."}`
- `{"data_url": "data:image/png;base64,..."}`
- `{"bytes_base64": "...", "mime_type": "image/png"}`


In [ ]:
REFERENCE_IMAGE_PATH = r'F:\Documents\GitHub\AI_TRPG_616\test\test.png'
reference_path = Path(REFERENCE_IMAGE_PATH)

if not reference_path.exists():
    print(f'参考图不存在：{reference_path}')
else:
    reference_result = generate_image(
        prompt='在保留参考图主体特征的前提下，转成赛博霓虹风格，增加发光装置与未来都市背景',
        trigger='manual',
        theme='neon',
        scene_id='notebook-reference-image-test',
        reference_images=[reference_path],
        save_output=True,
        use_cache=True,
    )

    print('参考图数量 =', reference_result.reference_count)
    print('最终 prompt =', reference_result.prompt)
    print('输出文件路径 =', reference_result.output_path)

    if reference_result.output_path:
        display(Image(filename=str(reference_result.output_path)))


## 2. 前后端联调用法：请求本地 HTTP 接口

这部分对应前端实际调用图片服务的方式。

先在项目根目录启动本地服务：

`python Code/image_model/function_ImageGeneration.py`

然后把 `USE_HTTP_SERVER = True`，再执行下面这一格。


In [ ]:
import requests

if not USE_HTTP_SERVER:
    print('已跳过 HTTP 联调测试。需要时把 USE_HTTP_SERVER 改成 True。')
else:
    response = requests.post(
        SERVER_ENDPOINT,
        json={
            'prompt': PROMPT,
            'trigger': TRIGGER,
            'theme': THEME,
            'scene_id': SCENE_ID,
            'characters': CHARACTERS,
            'reference_images': [],
        },
        timeout=120,
    )
    response.raise_for_status()
    data = response.json()
    print(data)
    if data.get('image_url'):
        display(Image(url=data['image_url']))
